# 🎯 [MNT10_M08] 동시 20개소 긴급 정비 요청 발생 모의 출동 스케줄링 스트레스 테스트
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT10/mnt10_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT10_M08 |
| **미션 주제** | 동시 20개소 긴급 정비 요청 발생 모의 출동 스케줄링 스트레스 테스트 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
야전에서 장비가 고장 나면 이동정비반이 얼마나 빨리 도착해 고치느냐가 부대 전투력 회복 속도를 좌우하지만, 가까운 정비반부터 보내는 방식은 요청이 몰리는 지역의 정비반에 부하를 집중시키고 핵심 장비를 뒤로 미루는 문제가 있다. 본 프로젝트는 정비인력의 부하율과 공정 균형을 데이터로 예측·관리하는 것을 목표로, 교육용 가상 지역의 긴급 정비 요청과 이동정비반 자료를 코드로 만들어 분석한다. 요청 데이터 스키마 정의, 정비반 상태 자료 정제, 거리 기준 Baseline과 탐색적 분석, 중요도·정비 소요시간·이동 경로를 종합한 우선순위 엔진, 출동 지연 위험도와 현장 수리 가능성 점수, 관제 대시보드와 설명가능성, 정비대대장 승인(HITL) 시뮬레이터, 동시 다발 요청 스트레스 시험, AAR 가중치 재튜닝과 효과 산출, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 핵심 장비의 가동중단 시간을 줄이면서 정비반 간 부하를 고르게 하고, 출동 순서 변경과 재배치는 지휘관이 근거를 보고 결정하도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 산출물(deliverables) 디렉토리 구성 안내

본 과제는 실무 현장에서 바로 쓰일 수 있는 국방 소프트웨어 아키텍처 표준을 지향합니다.  
과제를 진행하며 생성되는 산출물은 아래 규칙에 따라 각 폴더에 체계적으로 저장해 주세요.

| 디렉토리 | 군수 실무 비유 | 주 역할 및 산출물 파일 | 분리 목적 |
| :--- | :--- | :--- | :--- |
| **`docs/`** | 야전 교범 / 정비 지침서 | `mnt10_logistics_domain_spec.md` (미션1), `mnt10_operations_guide.md` (미션10) | 코드를 모르는 지휘관/후임자도 데이터 규격과 비상 매뉴얼을 이해하도록 보장 |
| **`notebooks/`** | 정비 시험실 / 연구실 | `mnt10_eda_analysis.ipynb` (미션3), 본 실습 노트북 | AI와 대화하며 자유롭게 데이터를 시각화하고 알고리즘을 반복 실험 |
| **`src/`** | 실제 장착 핵심 부품 모듈 | `src/etl/`, `src/features/`, `src/models/`, `src/dashboard/`, `src/optimization/` | 운영 서버에서 자동 배치 실행 가능한 순수 파이썬 모듈(`.py`)로 부품화 |
| **`reports/`** | 지휘관 결심 보고서 | `mnt10_roi_analysis.md` (미션9), AAR 오배정 분석서 | 정비대대장/군수참모에게 정비 지연 15% 감축 등 정량 ROI 성과 보고 |
| **`pipeline/`** | 자동화 컨베이어 벨트 | `pipeline/mnt10_retrain.py` (미션10) | 긴급 요청 유입 시 전처리부터 관제까지 자동 구동 및 재학습 파이프라인 |

> **💡 [바이브 코딩(AI 협업) 실습 요령]**  
> 1. **노트북에서 먼저 검증**: 본 노트북의 각 미션 셀에서 AI에게 질의하며 단계별 코드를 실행하고 결과를 확인합니다.  
> 2. **AI에게 모듈화 요청**: 검증 완료 후 AI에게 *"이 코드를 `src/.../` 파일에 넣을 수 있게 함수/클래스로 모듈화해줘"*라고 요청합니다.  
> 3. **디렉토리별 저장**: 제안된 코드를 해당 파일에 저장하여 자연스럽게 실무 패키지를 완성합니다.

---



In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 동시 20개소 긴급 정비 요청 발생 모의 출동 스케줄링 스트레스 테스트
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

동시 20개소 긴급 정비 요청 발생 모의 스트레스 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **동시 20개소 긴급 정비 요청이 한꺼번에 들어오는 상황에서 출동 스케줄링과 관제 화면이 제때 동작하는지 스트레스 시험한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **동시 20·40·60개소 요청, 정비반 1개 출동 불가, 손상 요청 데이터를 주입하고 할당 최적화로 배정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/dashboard/mnt10_app.py (Streamlit/Dash 등)` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 의사결정 지원에 최적화된 시각적 위계 및 사용자 경험(UX)

#### 미션 PASS 조건

- 정량 기준: 대시보드 로딩 시간 2초 이내, 경보 발령 정확도 95% 이상


- 진행 기준: 웹 기반 분석 대시보드 구동 및 실시간 필터링 기능 검증


- 검증 방법: 실제 군수 데이터 연동 화면 테스트 및 UI 동작 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""동시 20개소(최대 60개소) 긴급 정비 요청 발생, 정비반 1개 출동 불가, 손상 요청 데이터를 주입해 출동 스케줄링과 관제 대시보드를 스트레스 테스트한다. 스케줄링+화면 생성 시간, '핵심장비 4시간 내 도착 불가' 경보 정확도, 요청 유실, 손상 데이터의 수동 확인 분리와 필터 조회 응답을 검증한다."""
import time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

# ---------------------------------------------------------------------------
# [교육용가상자료] 야전 이동정비반·긴급 정비 요청 생성기 (실제 부대·위치·장비·인원 정보 없음)
# 60km x 60km 가상 지역에 이동정비반 6개와 30일간의 긴급 정비 요청을 만든다.
# 장비 구분(mobility/fires/comms/power)과 고장 유형은 교육용 추상 분류다.
# ---------------------------------------------------------------------------
AREA_KM, DUTY_H = 60, 10.0                     # 가상 작전지역 크기, 정비반 1일 기준 가용시간
FAULT_H = {"engine": 6.0, "drivetrain": 5.0, "hydraulic": 3.0, "electric": 2.0, "comm_module": 1.5, "cooling": 2.5}
FIELD_P = {"engine": 0.3, "drivetrain": 0.35, "hydraulic": 0.75, "electric": 0.9, "comm_module": 0.95, "cooling": 0.8}
EQUIP_F = {"mobility": 1.0, "fires": 1.2, "comms": 0.7, "power": 0.8}
FAULT_OF = {"mobility": [0.3, 0.25, 0.2, 0.1, 0.0, 0.15], "fires": [0.2, 0.15, 0.35, 0.15, 0.0, 0.15],
            "comms": [0.0, 0.0, 0.0, 0.4, 0.5, 0.1], "power": [0.35, 0.0, 0.0, 0.35, 0.0, 0.3]}
CRIT_P = {"mobility": [0.45, 0.35, 0.2], "fires": [0.2, 0.35, 0.45], "comms": [0.3, 0.4, 0.3], "power": [0.6, 0.3, 0.1]}
HUBS = [(15, 20), (45, 25), (30, 50)]           # 가상 훈련장 3곳(요청이 몰리는 지점)


def make_teams(seed=0):
    rng = np.random.default_rng(seed)
    base = [(8, 10), (30, 6), (52, 12), (10, 44), (32, 38), (54, 50)]
    skills = [["fires", "mobility"], ["comms", "mobility"], ["fires", "power"],
              ["mobility", "power"], ["comms", "fires"], ["comms", "mobility", "power"]]
    return [{"team": f"MT{i + 1}", "x": float(x), "y": float(y), "crew": int(rng.integers(3, 6)), "skills": s}
            for i, ((x, y), s) in enumerate(zip(base, skills))]


def make_requests(n_days=30, per_day=10, seed=1, surge_days=(8, 9, 10, 11, 22, 23, 24)):
    """하루 평균 per_day건, 훈련 주간(surge_days)에는 1.6배의 긴급 정비 요청을 만든다."""
    rng = np.random.default_rng(seed)
    rows = []
    for d in range(n_days):
        n = rng.poisson(per_day * (1.6 if d in surge_days else 1.0))
        weather = rng.choice(["clear", "rain", "snow"], p=[0.65, 0.25, 0.1])
        for _ in range(n):
            if rng.random() < 0.7:
                hx, hy = HUBS[rng.integers(0, 3)]
                x, y = np.clip(rng.normal(hx, 6), 0, AREA_KM), np.clip(rng.normal(hy, 6), 0, AREA_KM)
            else:
                x, y = rng.uniform(0, AREA_KM, 2)
            eq = str(rng.choice(list(EQUIP_F), p=[0.45, 0.25, 0.2, 0.1]))
            fault = str(rng.choice(list(FAULT_H), p=FAULT_OF[eq]))
            parts = bool(rng.random() < 0.75)
            bad = weather != "clear"
            std_h = FAULT_H[fault] * EQUIP_F[eq] * (1.35 if not parts else 1.0) * (1.15 if bad else 1.0) * rng.lognormal(0, 0.12)
            rows.append({"t_req": d * 24 + float(np.clip(rng.normal(13, 4), 5, 23)), "day": d,
                         "x": float(x), "y": float(y), "equip": eq, "fault": fault,
                         "crit": int(rng.choice([1, 2, 3], p=CRIT_P[eq])), "parts_ready": parts,
                         "weather": str(weather), "road": "unpaved" if y > 30 and rng.random() < 0.7 else "paved",
                         "field_ok": bool(rng.random() < FIELD_P[fault] * (1.0 if parts else 0.5)),
                         "std_repair_h": round(float(std_h), 2), "road_delay": float(rng.lognormal(0, 0.15))})
    df = pd.DataFrame(rows).sort_values("t_req").reset_index(drop=True)
    df.insert(0, "req_id", [f"R{i:04d}" for i in range(len(df))])
    return df
# ---------------------------------------------------------------------------

# ---- 출동 배정·정비 모의 공통 함수 ----
W0 = {"crit": 1.0, "wait": 0.3, "short": 0.2, "balance": 0.5}   # 우선순위·배정 기본 가중치(미션 9에서 재튜닝)


def travel_h(team, r, actual=False):
    """도로 굴곡 1.3배, 포장 35km/h·비포장 25km/h, 악천후 20% 감속. actual=True면 실제 도로 지연을 반영."""
    d = 1.3 * np.hypot(team["x"] - r["x"], team["y"] - r["y"])
    v = (35.0 if r["road"] == "paved" else 25.0) / (1.2 if r["weather"] != "clear" else 1.0)
    return d / v * (r["road_delay"] if actual else 1.0)


def on_site_h(team, r):
    """현장 정비시간: 4인 기준 표준시간을 투입 인원으로 환산. 현장 수리 불가면 진단·후송 조치 1시간."""
    return r["std_repair_h"] * 4 / team["crew"] if r["field_ok"] else 1.0


def priority(r, now, est_h, w):
    return w["crit"] * r["crit"] + w["wait"] * (now - r["t_req"]) + w["short"] / (1 + est_h)


def simulate(req, teams, policy="baseline", w=None, cycle_h=1.0, est=None):
    """배정 주기(cycle_h)마다 대기 요청을 정렬해 이동정비반에 배정하고 출동·정비·복귀를 모의한다.
    baseline: 접수 순서 + 거리 최근접 정비반 / priority: 우선순위 점수 순서 + 도착 예상시각·당일 부하 균형"""
    w = w or W0
    est = np.asarray(est if est is not None else req["std_repair_h"].values, float)
    n_days = int(req["day"].max()) + 1
    free = {t["team"]: 0.0 for t in teams}
    load = {t["team"]: np.zeros(n_days + 5) for t in teams}
    recs, out, pending, i, now = req.to_dict("records"), [], [], 0, cycle_h
    while i < len(recs) or pending:
        while i < len(recs) and recs[i]["t_req"] <= now:
            pending.append((i, recs[i])); i += 1
        if policy == "priority":
            pending.sort(key=lambda p: -priority(p[1], now, est[p[0]], w))
        for k, r in pending:
            cand = [t for t in teams if r["equip"] in t["skills"]]
            day = min(int(now // 24), n_days + 4)
            if policy == "baseline":
                tm = min(cand, key=lambda t: np.hypot(t["x"] - r["x"], t["y"] - r["y"]))
            else:
                tm = min(cand, key=lambda t: max(free[t["team"]], now) + travel_h(t, r)
                         + w["balance"] * load[t["team"]][day] / DUTY_H * 4)
            backlog = max(0.0, free[tm["team"]] - now)
            go = max(free[tm["team"]], now)
            tr = travel_h(tm, r, actual=True)
            work = on_site_h(tm, r)
            free[tm["team"]] = go + 2 * tr + work
            load[tm["team"]][min(int(go // 24), n_days + 4)] += 2 * tr + work
            out.append({**r, "idx": k, "team": tm["team"], "dist_km": float(np.hypot(tm["x"] - r["x"], tm["y"] - r["y"])),
                        "backlog_h": backlog, "cycle_wait_h": now - r["t_req"], "dispatch": go, "travel_h": tr,
                        "work_h": work, "response_h": go + tr - r["t_req"], "downtime_h": go + tr + work - r["t_req"]})
        pending, now = [], now + cycle_h
    L = pd.DataFrame({k: v[:n_days] for k, v in load.items()}) / DUTY_H      # 정비반별 일일 부하율
    return pd.DataFrame(out).sort_values("idx").reset_index(drop=True), L


def metrics(res, L):
    c3 = res[res["crit"] == 3]
    return {"평균 도착(h)": round(res["response_h"].mean(), 2), "핵심장비 도착(h)": round(c3["response_h"].mean(), 2),
            "핵심장비 90%(h)": round(c3["response_h"].quantile(0.9), 2), "평균 부하율": round(float(L.values.mean()), 3),
            "부하율 편차": round(float(L.mean().std()), 3), "과부하 일수": int((L > 1.0).values.sum()),
            "총 이동(h)": round(float(res["travel_h"].sum() * 2), 1)}

CW = {1: 1.0, 2: 3.0, 3: 6.0}
ALERT_H = 4.0            # 핵심장비(crit 3) 도착 목표시간
T0 = 15 * 24 + 10.0      # 15일차 10시 동시 다발 요청 발생


def make_burst(n, seed):
    """[교육용가상자료] 동시 n개소 요청(같은 시각 접수). 일부 정비반은 기존 작업으로 바쁜 상태."""
    pool = make_requests(n_days=10, seed=seed)
    b = pool.sample(n, random_state=seed).reset_index(drop=True).astype({"x": float, "crit": object})
    b["t_req"] = T0; b["req_id"] = [f"B{seed}-{i:02d}" for i in range(n)]
    return b


def validate(r):
    errs = []
    if not (0 <= r["x"] <= AREA_KM and 0 <= r["y"] <= AREA_KM):
        errs.append("좌표 범위 오류")
    if r["equip"] not in EQUIP_F or r["fault"] not in FAULT_H:
        errs.append("장비/고장 코드 오류")
    if r["crit"] not in (1, 2, 3):
        errs.append("중요도 누락")
    return errs


def schedule(burst, teams, busy_until):
    """할당 문제로 동시 요청을 (정비반, 순번)에 배정. 비용 = 중요도 가중 x 예상 도착시각."""
    slots = int(np.ceil(len(burst) / len(teams))) + 2
    cols = [(t, s) for t in teams for s in range(slots)]
    avg = 4.0
    cost = np.full((len(burst), len(cols)), 1e6)
    for i, r in enumerate(burst.to_dict("records")):
        for j, (t, s) in enumerate(cols):
            if r["equip"] in t["skills"]:
                cost[i, j] = CW[r["crit"]] * (busy_until[t["team"]] - T0 + s * avg + travel_h(t, r))
    ri, cj = linear_sum_assignment(cost)
    plan = sorted(((i, cols[j][0], cols[j][1]) for i, j in zip(ri, cj)), key=lambda p: (p[1]["team"], p[2]))
    free, rows = dict(busy_until), []
    for i, t, s in plan:
        r = burst.iloc[i]
        go = max(free[t["team"]], T0)
        est = go + travel_h(t, r) - T0
        act_tr = travel_h(t, r, actual=True)
        rows.append({"req_id": r["req_id"], "crit": r["crit"], "equip": r["equip"], "team": t["team"], "eta_h": est,
                     "actual_h": go + act_tr - T0, "alert": r["crit"] == 3 and est > ALERT_H})
        free[t["team"]] = go + 2 * act_tr + on_site_h(t, r)      # 실제로는 실측 이동시간대로 다음 출동이 정해진다
    return pd.DataFrame(rows)


def dashboard(plan, teams):
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    ld = plan.groupby("team").size().reindex([t["team"] for t in teams], fill_value=0)
    ax[0].bar(ld.index, ld.values); ax[0].set_title("Assigned requests per team")
    ax[1].scatter(plan["eta_h"], plan["crit"], c=np.where(plan["alert"], "red", "gray")); ax[1].axvline(ALERT_H, ls="--")
    ax[1].set_xlabel("ETA (h)"); ax[1].set_yticks([1, 2, 3]); ax[1].set_title("ETA by criticality (red = alert)")
    ax[2].hist(plan["eta_h"], bins=15); ax[2].set_title("ETA distribution")
    fig.tight_layout(); fig.canvas.draw()
    return fig


teams_all = make_teams()
rng = np.random.default_rng(4)
busy0 = {t["team"]: T0 + float(rng.uniform(0, 3)) for t in teams_all}       # 기존 작업 잔여시간
rows, keep = [], {}
for n, down, bad in [(20, None, 0), (20, "MT5", 3), (40, "MT5", 3), (60, "MT5", 5)]:
    burst = make_burst(n, seed=n)
    for k in range(bad):                                                     # 손상 데이터 주입
        burst.loc[k * 3, ["x", "equip", "crit"][k % 3]] = [999.0, "unknown", 0][k % 3]
    teams = [t for t in teams_all if t["team"] != down]
    t0 = time.perf_counter()
    ok = burst[[not validate(r) for r in burst.to_dict("records")]]
    manual = burst.drop(ok.index)
    plan = schedule(ok, teams, {t["team"]: busy0[t["team"]] for t in teams})
    fig = dashboard(plan, teams)
    sec = time.perf_counter() - t0
    truth = (plan["crit"] == 3) & (plan["actual_h"] > ALERT_H)
    acc = float((plan["alert"] == truth).mean())
    lost = n - len(plan) - len(manual)
    rows.append({"상황": f"{n}개소" + (f", {down} 불가" if down else "") + (f", 손상 {bad}건" if bad else ""), "배정": len(plan), "수동 확인": len(manual),
                 "유실": lost, "처리+화면(초)": round(sec, 2), "핵심장비 경보": int(plan["alert"].sum()), "경보 정확도": round(acc, 3),
                 "핵심 평균 도착(h)": round(plan.loc[plan["crit"] == 3, "eta_h"].mean(), 2)})
    keep[n, down] = (fig, plan)
    if (n, down) != (60, "MT5"):
        plt.close(fig)
T = pd.DataFrame(rows)
print("[교육용가상자료] 동시 다발 긴급 정비 요청 스트레스 테스트")
print(T.to_string(index=False))
m = T.iloc[1]
print(f"\n동시 20개소(정비반 1개 불가·손상 3건): 처리+화면 {m['처리+화면(초)']}초 (목표 2초 이내: {'달성' if T['처리+화면(초)'].max() <= 2 else '미달'}), "
      f"경보 정확도 {m['경보 정확도']:.1%} (목표 95% 이상: {'달성' if T['경보 정확도'].min() >= 0.95 else '미달'}), 유실 {int(T['유실'].sum())}건")
print("손상 요청은 삭제하지 않고 '수동 확인' 목록으로 넘겨 관제병이 전화·무전으로 재확인한다.")

plan = keep[20, "MT5"][1]
qt = []
for c in (1, 2, 3):
    for e in EQUIP_F:
        t0 = time.perf_counter(); plan[(plan["crit"] == c) & (plan["equip"] == e)]; qt.append((time.perf_counter() - t0) * 1000)
print(f"실시간 필터(중요도 x 장비 12조합) 평균 {np.mean(qt):.2f} ms")
plan = keep[40, "MT5"][1]
miss = plan[(plan["crit"] == 3) & (plan["actual_h"] > ALERT_H) & ~plan["alert"]]
print(f"40개소 상황: 놓친 경보 {len(miss)}건, 헛경보 {int((plan['alert'] & ~((plan['crit'] == 3) & (plan['actual_h'] > ALERT_H))).sum())}건"
      " -> 도로 지연(비포장·기상)을 ETA에 반영하면 줄일 수 있다.")
print("기능 축소 운용: 처리 시간이 2초를 넘으면 할당 최적화 대신 '중요도 순 + 최근접' 규칙으로 즉시 배정하고 최적화는 뒤에서 재계산한다.")
plt.show()


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 동시 다발 긴급 정비 요청 스트레스 테스트
                 상황  배정  수동 확인  유실  처리+화면(초)  핵심장비 경보  경보 정확도  핵심 평균 도착(h)
               20개소  20      0   0      0.24        1   1.000         3.17
20개소, MT5 불가, 손상 3건  17      3   0      0.23        3   1.000         4.56
40개소, MT5 불가, 손상 3건  37      3   0      0.23        3   0.973         4.55
60개소, MT5 불가, 손상 5건  55      5   0      0.23        6   1.000         5.64

동시 20개소(정비반 1개 불가·손상 3건): 처리+화면 0.23초 (목표 2초 이내: 달성), 경보 정확도 100.0% (목표 95% 이상: 달성), 유실 0건
손상 요청은 삭제하지 않고 '수동 확인' 목록으로 넘겨 관제병이 전화·무전으로 재확인한다.
실시간 필터(중요도 x 장비 12조합) 평균 0.69 ms
40개소 상황: 놓친 경보 0건, 헛경보 1건 -> 도로 지연(비포장·기상)을 ETA에 반영하면 줄일 수 있다.
기능 축소 운용: 처리 시간이 2초를 넘으면 할당 최적화 대신 '중요도 순 + 최근접' 규칙으로 즉시 배정하고 최적화는 뒤에서 재계산한다.
```
</details>
